In [0]:
%pip install httpx-sse httpx
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("secret_scope", "eventhub")
dbutils.widgets.text("secret_key", "eh-connection-string")
dbutils.widgets.text("eh_name", "roksolana-wikipedia-recentchange")
dbutils.widgets.text("wiki_filter", "en.wikipedia.org")
dbutils.widgets.text("max_events", "2000")
dbutils.widgets.text("retry_total", "3")
dbutils.widgets.text("retry_backoff_factor", "2")

SECRET_SCOPE = dbutils.widgets.get("secret_scope")
SECRET_KEY = dbutils.widgets.get("secret_key")
EH_NAME = dbutils.widgets.get("eh_name")
WIKI_FILTER = dbutils.widgets.get("wiki_filter")
MAX_EVENTS = int(dbutils.widgets.get("max_events"))
RETRY_TOTAL = int(dbutils.widgets.get("retry_total"))
RETRY_BACKOFF_FACTOR = int(dbutils.widgets.get("retry_backoff_factor"))

EH_CONN_STR = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)

In [0]:
import json
import asyncio
import httpx
from httpx_sse import aconnect_sse
from azure.eventhub.aio import EventHubProducerClient
from azure.eventhub import EventData

FIELDS = ["id", "type", "title", "user", "bot", "minor", "timestamp", "wiki", "server_name", "length"]

async def run():
    producer = EventHubProducerClient.from_connection_string(
        conn_str=EH_CONN_STR,
        eventhub_name=EH_NAME,
        retry_total=RETRY_TOTAL,
        retry_backoff_factor=RETRY_BACKOFF_FACTOR,
    )

    headers = {"User-Agent": "DatabricksLab3Producer/1.0 (roksolana.shendiu770@softserve.academy)"}
    sent_count = 0

    async with producer:
        batch = await producer.create_batch()
        async with httpx.AsyncClient(headers=headers, timeout=None) as http_client:
            async with aconnect_sse(http_client, "GET", "https://stream.wikimedia.org/v2/stream/recentchange") as event_source:
                async for sse in event_source.aiter_sse():
                    if not sse.data:
                        continue
                    record = json.loads(sse.data)
                    if record.get("server_name") != WIKI_FILTER:
                        continue

                    payload = {k: record.get(k) for k in FIELDS}

                    try:
                        batch.add(EventData(json.dumps(payload)))
                    except ValueError:
                        asyncio.create_task(producer.send_batch(batch))
                        batch = await producer.create_batch()
                        batch.add(EventData(json.dumps(payload)))

                    sent_count += 1
                    if sent_count >= MAX_EVENTS:
                        break

        if len(batch) > 0:
            await producer.send_batch(batch)

asyncio.run(run())